# Moirai Demo — Chạy thẳng code từ HuggingFace

Nguồn: https://huggingface.co/Pranavv/moirai-base

## Setup (chạy 1 lần)
```bash
git clone https://github.com/SalesforceAIResearch/uni2ts.git
cd uni2ts
pip install -e '.[notebook]'
```
Hoặc đơn giản hơn:
```bash
pip install uni2ts gluonts torch huggingface_hub
```

In [ ]:
import torch
import pandas as pd
from gluonts.dataset.pandas import PandasDataset
from gluonts.dataset.split import split
from huggingface_hub import hf_hub_download

from uni2ts.eval_util.plot import plot_single
from uni2ts.model.moirai import MoiraiForecast

print(f"PyTorch: {torch.__version__}")
print(f"GPU available: {torch.cuda.is_available()}")

## Cấu hình

In [ ]:
SIZE = "base"   # {'small' (14M), 'base' (91M), 'large' (311M)}
PDT  = 20       # prediction length
CTX  = 200      # context length
PSZ  = "auto"   # patch size: "auto" hoặc 8, 16, 32, 64, 128
BSZ  = 32       # batch size
TEST = 100      # độ dài test set

## Load dữ liệu mẫu (demo dataset)

In [ ]:
url = (
    "https://gist.githubusercontent.com/rsnirwan/c8c8654a98350fadd229b00167174ec4"
    "/raw/a42101c7786d4bc7695228a0f2c8cea41340e18f/ts_wide.csv"
)
df = pd.read_csv(url, index_col=0, parse_dates=True)
print(f"Shape: {df.shape}")
df.head()

In [ ]:
# Visualize nhanh
df.plot(figsize=(14, 4), legend=False, title="Demo dataset — tất cả series")
import matplotlib.pyplot as plt
plt.tight_layout()
plt.show()

## Chuẩn bị GluonTS dataset + train/test split

In [ ]:
ds = PandasDataset(dict(df))

train, test_template = split(ds, offset=-TEST)

test_data = test_template.generate_instances(
    prediction_length=PDT,
    windows=TEST // PDT,
    distance=PDT,
)
print(f"Số series: {len(df.columns)}")
print(f"Test windows mỗi series: {TEST // PDT}")

## Load Moirai pre-trained model

Lần đầu sẽ download checkpoint (~370MB cho base). Các lần sau dùng cache.

In [ ]:
device = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"Dùng device: {device}")

model = MoiraiForecast.load_from_checkpoint(
    checkpoint_path=hf_hub_download(
        repo_id=f"Salesforce/moirai-1.0-R-{SIZE}",
        filename="model.ckpt",
    ),
    prediction_length=PDT,
    context_length=CTX,
    patch_size=PSZ,
    num_samples=100,
    target_dim=1,
    feat_dynamic_real_dim=ds.num_feat_dynamic_real,
    past_feat_dynamic_real_dim=ds.num_past_feat_dynamic_real,
    map_location=device,
)
print("Model loaded OK.")

## Chạy dự báo (zero-shot inference)

In [ ]:
predictor = model.create_predictor(batch_size=BSZ)
forecasts = list(predictor.predict(test_data.input))
print(f"Số forecasts: {len(forecasts)}")
print(f"Forecast shape (samples x horizon): {forecasts[0].samples.shape}")

## Visualize kết quả dự báo

In [ ]:
import matplotlib.pyplot as plt

input_it    = iter(test_data.input)
label_it    = iter(test_data.label)
forecast_it = iter(forecasts)

# Hiển thị 3 ví dụ đầu
for i in range(3):
    inp      = next(input_it)
    label    = next(label_it)
    forecast = next(forecast_it)

    fig, ax = plt.subplots(figsize=(12, 4))
    plot_single(
        inp, label, forecast,
        context_length=CTX,
        name=f"Window {i+1}",
        show_label=True,
        ax=ax,
    )
    plt.title(f"Moirai {SIZE.upper()} — Zero-shot forecast (window {i+1})")
    plt.tight_layout()
    plt.show()

## Tính metric đánh giá

In [ ]:
from gluonts.evaluation import Evaluator

# Re-generate test data để lấy lại iterator
test_data2 = test_template.generate_instances(
    prediction_length=PDT,
    windows=TEST // PDT,
    distance=PDT,
)
forecasts2 = list(predictor.predict(test_data2.input))

evaluator = Evaluator(quantiles=[0.1, 0.5, 0.9])
agg_metrics, _ = evaluator(
    ts_iterator=(entry["target"] for entry in test_data2.label),
    fcst_iterator=iter(forecasts2),
)

print("=== Kết quả Moirai Zero-Shot ===")
for key in ["MSE", "MAE", "MASE", "MSIS", "mean_wQuantileLoss"]:
    val = agg_metrics.get(key)
    if val is not None:
        print(f"  {key:25s}: {val:.4f}")

Demo hoàn tất. Xem notebook `run_own_data.ipynb` để chạy với CSV tài chính của bạn.